# Streamlit

Streamlit 应用就是一个从上到下执行的 Python 脚本。用户改动控件时，脚本会重新运行，因此耗时的数据加载和模型初始化要做缓存。


In [ ]:
def build_streamlit_app_code() -> str:
    return '''
import pandas as pd
import streamlit as st


st.set_page_config(page_title="Sales Dashboard", layout="wide")
st.title("Sales Dashboard")


@st.cache_data
def load_sales() -> pd.DataFrame:
    return pd.DataFrame(
        [
            {"region": "East", "product": "A", "revenue": 1200},
            {"region": "East", "product": "B", "revenue": 900},
            {"region": "West", "product": "A", "revenue": 1600},
            {"region": "West", "product": "B", "revenue": 700},
        ]
    )


sales = load_sales()
regions = st.sidebar.multiselect("Region", sorted(sales["region"].unique()), default=sorted(sales["region"].unique()))
filtered = sales[sales["region"].isin(regions)]

total_revenue = int(filtered["revenue"].sum())
st.metric("Total revenue", f"${total_revenue:,}")

left, right = st.columns(2)
with left:
    st.subheader("Raw data")
    st.dataframe(filtered, use_container_width=True)

with right:
    st.subheader("Revenue by product")
    chart_data = filtered.groupby("product", as_index=False)["revenue"].sum()
    st.bar_chart(chart_data, x="product", y="revenue")
'''.strip()


print(build_streamlit_app_code())


## 状态和缓存

`st.session_state` 保存当前浏览器会话里的状态，适合计数器、聊天历史、筛选条件等用户级状态。`st.cache_data` 适合缓存可序列化数据结果，`st.cache_resource` 适合缓存数据库连接、模型对象这类资源。


In [ ]:
def build_streamlit_state_code() -> str:
    return '''
from dataclasses import dataclass

import streamlit as st


@dataclass
class TinyModel:
    name: str

    def predict(self, text: str) -> str:
        return f"{self.name}: {len(text)} chars"


@st.cache_resource
def load_model() -> TinyModel:
    return TinyModel(name="demo-model")


model = load_model()
st.title("Session State Demo")

if "history" not in st.session_state:
    st.session_state.history = []

text = st.text_input("Text")
if st.button("Predict") and text:
    result = model.predict(text)
    st.session_state.history.append({"text": text, "result": result})

st.write(st.session_state.history)
'''.strip()


print(build_streamlit_state_code())
